# Task 1 — Character-level GPT from scratch (Nikhil)

Executed record of the full run. All logic lives in the modules next to this notebook:

| File | Contents |
|---|---|
| `data.py` | text cleaning, `char_to_idx` / `idx_to_char`, story-level train/val split, fixed-length (x, y) windows |
| `model.py` | hand-written causal multi-head self-attention, pre-LN blocks, token + positional embeddings, LM head |
| `train.py` | AdamW, linear warm-up + cosine LR schedule, per-epoch eval, checkpoints, every Task 1 metric |
| `gen_metrics.py` | Distinct-n and repeated 4-gram rate |
| `configs/gpt_nikhil.yaml` | every hyperparameter for this run |

Raw data: `python task1_llm/data/download_tinystories.py` (run from repo root).

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

SRC = Path.cwd()
assert (SRC / "train.py").exists(), "run this notebook from task1_llm/member_nikhil/src"
sys.path.insert(0, str(SRC))
import train
CONFIG = SRC / "configs" / "gpt_nikhil.yaml"
print(CONFIG.read_text())

## Model definition (no prebuilt attention modules)

In [ ]:
import inspect, model
print(inspect.getsource(model.CausalSelfAttention))

## Train (12 epochs) — the raw log is also written to `reproducibility/raw_logs/nikhil/`

In [ ]:
result = train.run(CONFIG)
out = result["out_dir"]
print(result["run_id"])

## Metrics

In [ ]:
pd.set_option("display.max_colwidth", 80)
pd.read_csv(out / "metrics.csv")

In [ ]:
pd.read_csv(out / "epoch_metrics.csv").round(4)

## Loss curves, gradient norm and LR schedule

In [ ]:
display(Image(out / "loss_curves.png"))
display(Image(out / "grad_norm_and_lr.png"))

## Generated samples (greedy and temperature 0.8)

In [ ]:
print((out / "samples.txt").read_text())